In [1]:
!pip install torch transformers

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import SGDClassifier
import random
import string

class TCAV:
    """ Class for concept activation vectors for Keras models.

    Attributes:
        model: a roBerta LLM loaded trough huggingface API
        tokenizer: a tokenizer for roBerta
        cav: A numpy array containing the concept activation vector
        sensitivity: A numpy array containing sensitivities
        y_labels: A numpy array containing class labels
        bottleneck: a int indicating to which hidden layer extract the activations
    """

    def __init__(self, model=None, tokenizer=None, fix_length=None):
        """ Inizializza la classe con variabili vuote """
        self.model = model
        self.tokenizer = tokenizer
        self.cav = None
        self.sensitivity = None
        self.y_labels = None
        self.bottleneck = None
        self.model_activations = {} #where to store activations
        self.fix_length = fix_length
        return
    
    def forward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["forward_"+name] = []
        def fn(module, input, output):
            # print("leaf",output.is_leaf)
            # x = output
            # print("leaf",output[0].is_leaf)
            x = output[0]
            x.requires_grad_(True)
            x.retain_grad()
            self.model_activations["forward_"+name].append(x)
            # print("extracted activations", output.shape)
            # print("extracted activations", output[0].shape)
            # print("----------------",output[0].shape[1],"---------------")
            # self.fix_length = output[0].shape[1]
            return
        return fn
    
    def backward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["backward_"+name] = []
            
        def fn(module, grad_input, grad_output):
            # print("leaf",grad_output[0].is_leaf)
            
            self.model_activations["backward_"+name] = grad_output[0]
            # print("extracted grads", grad_output[0].shape)
            return
        
        return fn

    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(self.model.roberta.encoder.layer):
            raise ValueError("Invalid layer for sampling")
        
        # layers = list(self.model.children())
        # print(layers)
        self.bottleneck = str(bottleneck)   
        
        # self.model.classifier.dense.register_forward_hook(self.hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_forward_hook(self.forward_hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_full_backward_hook(self.backward_hook_fn(str(bottleneck)))
        return


    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        # print(len(inputs))
        if self.tokenizer is not None:
            if self.fix_length:
                x = self.tokenizer(inputs, return_tensors="pt", padding="max_length", max_length=self.fix_length, truncation=True)
            else:
                x = self.tokenizer(inputs, return_tensors="pt", padding="longest")
                self.fix_length = x["input_ids"].shape[1]
            # print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        x_train_concept = self._tokenize(tmp)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            _ = self.model(**x_train_concept)
            # print("attentions dimensions:", len(self.model_activations["forward_"+self.bottleneck][0].shape))
            if(len(self.model_activations["forward_"+self.bottleneck][0].shape)>2):
                concept_activations = self.model_activations["forward_"+self.bottleneck][0].reshape(self.model_activations["forward_"+self.bottleneck][0].shape[0],-1)
            else:
                concept_activations = self.model_activations["forward_"+self.bottleneck][0]
            # print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)
        ### Concatenate token by token
        
        # Train linear classifier
        lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)
        lm.fit(concept_activations.detach().numpy(), y_train_concept.numpy())
        self.cav = -lm.coef_.T
        # print("cav", len(self.cav))
        
        self.model_activations["forward_"+self.bottleneck] = [] #once calculated all results, reset for next operations                           
        self.model_activations["backward_"+self.bottleneck] = []
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        Versione PyTorch della funzione, con commenti che rimandano
        ai passaggi originali in Keras.
        """
        
        print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)

        activations = self.model_activations["forward_"+self.bottleneck][0].reshape(self.model_activations["forward_"+self.bottleneck][0].shape[0],-1) # Prendi l'ultima attivazione del bottleneck
        # print("output logits", output.logits.shape)
        # print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        # print("loss", loss)
        # print("activations",activations.is_leaf, activations)
        # print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = self.model_activations["backward_"+self.bottleneck]
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        # print("grads", grads)
        # concatenate grads
        grads = grads.reshape(grads.shape[0],-1)

        # Scalar product
        cav_tensor = self.cav
        
        # print("shapes", cav_tensor.shape, grads.shape)
        sensitivities = []
        for g in grads:
            sensitivities.append(np.dot(g, cav_tensor))
        
        sensitivity = np.array(sensitivities)
        # print("sensitivity", sensitivity)

        # Saving sensitivity
        self.sensitivity = sensitivity
        self.y_labels    = y_train.detach().cpu().numpy().reshape(-1)

        return
        
    def print_sensitivity(self, id_to_labels):
        """ Print sensitivity in a readable way """
        if isinstance(self.y_labels, list):
            self.y_labels = np.array(self.y_labels)

        num_labels = len(np.unique(self.y_labels))
        # print(num_labels)

        for label_idx in range(num_labels):
            value = np.sum(self.sensitivity[np.where(self.y_labels == label_idx)[0]] > 0) / np.where(self.y_labels == label_idx)[0].shape[0]
            print(f"Sensitivity for label {id_to_labels[label_idx]} is: {value:.2f}")
            
        return

/home/students/fmarmello/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/students/fmarmello/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
import json
import os
import torch
CLASSES = {'go': 0, 'java': 1, 'javascript': 2, 'php': 3, 'python': 4, 'ruby': 5}
INV_CLASSES = ['go', 'java', 'javascript', 'php', 'python', 'ruby']
CONCEPTS= ["comments", "function_declarations", "go_function_declarations", "java_function_declarations", "javascript_function_declarations", "php_function_declarations","python_function_declarations", "ruby_function_declarations"]
CLASSES_TO_EXAMINE = ['go', 'java', 'javascript', 'php', 'python', 'ruby']
MODEL_NAME = "huggingface/CodeBERTa-language-id"

data = []
with open(f"./data/code_classification/random_concept_dataset.jsonl", "r") as f:
    rnd = [json.loads(line) for line in f]
    for x in rnd:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})
        
TEST = ([d["text"] for d in data], [d["label"] for d in data])

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES_TO_EXAMINE))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Some weights of the model checkpoint at huggingface/CodeBERTa-language-id were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
concept = CONCEPTS[0]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 400 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=400)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for comments concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.00
Sensitivity for label java is: 0.98
Sensitivity for label javascript is: 0.27
Sensitivity for label php is: 0.07
Sensitivity for label python is: 0.88
Sensitivity for label ruby is: 0.22
--------------- Calculating TCAVs for comments concept with 400 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.00
Sensitivity for label java is: 0.74
Sensitivity for label javascript is: 0.41
Sensitivity for label php is: 0.04
Sensitivity for label python is: 0.87
Sensitivity for label ruby is: 0.50


In [6]:
concept = CONCEPTS[1]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.00
Sensitivity for label java is: 0.12
Sensitivity for label javascript is: 0.52
Sensitivity for label php is: 0.40
Sensitivity for label python is: 0.53
Sensitivity for label ruby is: 0.00
--------------- Calculating TCAVs for function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.12
Sensitivity for label java is: 0.41
Sensitivity for label javascript is: 0.11
Sensitivity for label php is: 0.29
Sensitivity for label python is: 1.00
Sensitivity for label ruby is: 0.01


In [7]:
concept = CONCEPTS[2]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for go_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 1.00
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 0.72
Sensitivity for label php is: 0.77
Sensitivity for label python is: 0.83
Sensitivity for label ruby is: 0.22
--------------- Calculating TCAVs for go_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 1.00
Sensitivity for label java is: 0.03
Sensitivity for label javascript is: 0.75
Sensitivity for label php is: 0.38
Sensitivity for label python is: 0.83
Sensitivity for label ruby is: 0.34


In [8]:
concept = CONCEPTS[3]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for java_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.16
Sensitivity for label java is: 1.00
Sensitivity for label javascript is: 0.08
Sensitivity for label php is: 0.33
Sensitivity for label python is: 0.26
Sensitivity for label ruby is: 0.17
--------------- Calculating TCAVs for java_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.12
Sensitivity for label java is: 1.00
Sensitivity for label javascript is: 0.18
Sensitivity for label php is: 0.15
Sensitivity for label python is: 0.06
Sensitivity for label ruby is: 0.01


In [9]:
concept = CONCEPTS[4]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for javascript_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.04
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 1.00
Sensitivity for label php is: 0.26
Sensitivity for label python is: 0.08
Sensitivity for label ruby is: 0.01
--------------- Calculating TCAVs for javascript_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.03
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 1.00
Sensitivity for label php is: 0.34
Sensitivity for label python is: 0.06
Sensitivity for label ruby is: 0.01


In [10]:
concept = CONCEPTS[5]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for php_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.06
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 0.63
Sensitivity for label php is: 1.00
Sensitivity for label python is: 0.17
Sensitivity for label ruby is: 0.05
--------------- Calculating TCAVs for php_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.03
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 0.52
Sensitivity for label php is: 1.00
Sensitivity for label python is: 0.16
Sensitivity for label ruby is: 0.03


In [5]:
concept = CONCEPTS[6]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=400)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for python_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.25
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 0.64
Sensitivity for label php is: 0.36
Sensitivity for label python is: 1.00
Sensitivity for label ruby is: 0.21
--------------- Calculating TCAVs for python_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.54
Sensitivity for label java is: 0.04
Sensitivity for label javascript is: 0.58
Sensitivity for label php is: 0.51
Sensitivity for label python is: 1.00
Sensitivity for label ruby is: 0.43


In [6]:
concept = CONCEPTS[7]
data = []
with open(f"./data/code_classification/{concept}_dataset.jsonl", "r") as f:
    concept_examples = [json.loads(line) for line in f]

    for x in concept_examples:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})

    print(f"--------------- Calculating TCAVs for {concept} concept with variable token number -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

    print(f"--------------- Calculating TCAVs for {concept} concept with 500 tokens fixed -------------")
    tcav_object = TCAV(model=model, tokenizer=tokenizer, fix_length=500)
    tcav_object.split_model(4)
    tcav_object.train_cav([d["text"] for d in data])
    tcav_object.calculate_sensitivity(TEST[0], TEST[1])
    tcav_object.print_sensitivity(INV_CLASSES)

--------------- Calculating TCAVs for ruby_function_declarations concept with variable token number -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.36
Sensitivity for label java is: 0.01
Sensitivity for label javascript is: 0.77
Sensitivity for label php is: 0.31
Sensitivity for label python is: 1.00
Sensitivity for label ruby is: 0.55
--------------- Calculating TCAVs for ruby_function_declarations concept with 500 tokens fixed -------------
calculating cavs
calculating sensitivity
Sensitivity for label go is: 0.40
Sensitivity for label java is: 0.00
Sensitivity for label javascript is: 0.66
Sensitivity for label php is: 0.51
Sensitivity for label python is: 1.00
Sensitivity for label ruby is: 0.57
